In [1]:
from sklearn.datasets import fetch_20newsgroups
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
import copy

print("="*80)
print("COMPARISON: TRADITIONAL FINE-TUNING vs LORA vs QLORA")
print("20 Newsgroups Dataset")
print("="*80)

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-03 02:17:49.950668: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-03 02:17:55.744510: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 02:18:02.994907: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sli

COMPARISON: TRADITIONAL FINE-TUNING vs LORA vs QLORA
20 Newsgroups Dataset


In [2]:
# ============================================================================
# Load 20 Newsgroups Dataset
# ============================================================================

print("\n[1/7] Loading 20 Newsgroups dataset...")

categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']

train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f"✓ Dataset loaded")
print(f"  Categories: {categories}")
print(f"  Train samples: {len(train_data.data)}")
print(f"  Test samples: {len(test_data.data)}")


[1/7] Loading 20 Newsgroups dataset...
✓ Dataset loaded
  Categories: ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
  Train samples: 2034
  Test samples: 1353


In [3]:
# ============================================================================
# Prepare Tokenizer
# ============================================================================

print("\n[2/7] Preparing tokenizer...")

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

print("✓ Tokenizer ready")



[2/7] Preparing tokenizer...
✓ Tokenizer ready


In [4]:
# ============================================================================
# Create Dataset Class
# ============================================================================

print("\n[3/7] Creating dataset class...")

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

# Create datasets
train_dataset = NewsDataset(train_data.data, train_data.target.tolist(), tokenizer)
test_dataset = NewsDataset(test_data.data, test_data.target.tolist(), tokenizer)

print(f"✓ Datasets created")


[3/7] Creating dataset class...
✓ Datasets created


In [5]:
# ============================================================================
# Define Metrics
# ============================================================================

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [6]:
# ============================================================================
# METHOD 1: TRADITIONAL FULL FINE-TUNING
# ============================================================================

print("\n" + "="*80)
print("METHOD 1: TRADITIONAL FULL FINE-TUNING")
print("="*80)

print("\n[4/7] Loading model for traditional fine-tuning...")

# Load fresh model
model_traditional = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=len(categories)
)
model_traditional.config.pad_token_id = tokenizer.pad_token_id

# Count parameters
total_params = sum(p.numel() for p in model_traditional.parameters())
trainable_params_traditional = sum(p.numel() for p in model_traditional.parameters() if p.requires_grad)

print(f"✓ Model loaded for traditional fine-tuning")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params_traditional:,} (100%)")
print(f"  ALL layers will be updated during training")

# Training arguments for traditional fine-tuning
training_args_traditional = TrainingArguments(
    output_dir='./results_traditional',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,  # Lower LR for full fine-tuning
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    logging_steps=50,
)

trainer_traditional = Trainer(
    model=model_traditional,
    args=training_args_traditional,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting traditional fine-tuning...")
start_time_traditional = time.time()

train_result_traditional = trainer_traditional.train()

end_time_traditional = time.time()
traditional_train_time = end_time_traditional - start_time_traditional

# Evaluate
results_traditional = trainer_traditional.evaluate()

print("\n✓ Traditional fine-tuning complete!")
print(f"  Training time: {traditional_train_time:.2f}s")
print(f"  Test accuracy: {results_traditional['eval_accuracy']:.4f}")
print(f"  Test F1: {results_traditional['eval_f1']:.4f}")

# Save model size
model_traditional.save_pretrained('./results_traditional/model')
import os
traditional_model_size = sum(
    os.path.getsize(os.path.join('./results_traditional/model', f))
    for f in os.listdir('./results_traditional/model')
    if os.path.isfile(os.path.join('./results_traditional/model', f))
) / (1024**2)

print(f"  Model checkpoint size: {traditional_model_size:.2f} MB")


METHOD 1: TRADITIONAL FULL FINE-TUNING

[4/7] Loading model for traditional fine-tuning...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded for traditional fine-tuning
  Total parameters: 124,442,880
  Trainable parameters: 124,442,880 (100%)
  ALL layers will be updated during training


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")



Starting traditional fine-tuning...


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.641500,0.627282,0.738359,0.778058,0.738359,0.734131
2,0.430500,0.663683,0.771619,0.774115,0.771619,0.770066
3,0.221200,0.757432,0.778271,0.778290,0.778271,0.777516


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



✓ Traditional fine-tuning complete!
  Training time: 6769.82s
  Test accuracy: 0.7783
  Test F1: 0.7775
  Model checkpoint size: 474.73 MB


In [7]:
# ============================================================================
# METHOD 2: LORA (LOW-RANK ADAPTATION)
# ============================================================================

print("\n" + "="*80)
print("METHOD 2: LORA (LOW-RANK ADAPTATION)")
print("="*80)

print("\n[5/7] Loading model for LoRA fine-tuning...")

# Load fresh model
model_lora = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=len(categories)
)
model_lora.config.pad_token_id = tokenizer.pad_token_id

# Configure LoRA
lora_config = LoraConfig(
    r=8,                                    # Rank of LoRA matrices
    lora_alpha=32,                          # Scaling factor
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["c_attn"],              # Apply to attention layers
    inference_mode=False,
)

# Apply LoRA
model_lora = get_peft_model(model_lora, lora_config)

print(f"✓ LoRA applied to model")
model_lora.print_trainable_parameters()

trainable_params_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
frozen_params_lora = sum(p.numel() for p in model_lora.parameters() if not p.requires_grad)

print(f"\n  Total parameters: {total_params:,}")
print(f"  Trainable (LoRA): {trainable_params_lora:,} ({100*trainable_params_lora/total_params:.2f}%)")
print(f"  Frozen (base): {frozen_params_lora:,}")

# Training arguments for LoRA
training_args_lora = TrainingArguments(
    output_dir='./results_lora',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,  # Higher LR works well with LoRA
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    logging_steps=50,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting LoRA fine-tuning...")
start_time_lora = time.time()

train_result_lora = trainer_lora.train()

end_time_lora = time.time()
lora_train_time = end_time_lora - start_time_lora

# Evaluate
results_lora = trainer_lora.evaluate()

print("\n✓ LoRA fine-tuning complete!")
print(f"  Training time: {lora_train_time:.2f}s")
print(f"  Test accuracy: {results_lora['eval_accuracy']:.4f}")
print(f"  Test F1: {results_lora['eval_f1']:.4f}")

# Save model size (only LoRA adapters)
model_lora.save_pretrained('./results_lora/model')
lora_model_size = sum(
    os.path.getsize(os.path.join('./results_lora/model', f))
    for f in os.listdir('./results_lora/model')
    if os.path.isfile(os.path.join('./results_lora/model', f))
) / (1024**2)

print(f"  LoRA adapter size: {lora_model_size:.2f} MB")


METHOD 2: LORA (LOW-RANK ADAPTATION)

[5/7] Loading model for LoRA fine-tuning...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


✓ LoRA applied to model
trainable params: 297,984 || all params: 124,740,864 || trainable%: 0.2389

  Total parameters: 124,442,880
  Trainable (LoRA): 297,984 (0.24%)
  Frozen (base): 124,442,880

Starting LoRA fine-tuning...


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.675800,0.687145,0.715447,0.730394,0.715447,0.695810
2,0.522900,0.549615,0.757576,0.751770,0.757576,0.753871
3,0.405000,0.559043,0.764228,0.760351,0.764228,0.760863


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



✓ LoRA fine-tuning complete!
  Training time: 5613.52s
  Test accuracy: 0.7642
  Test F1: 0.7609
  LoRA adapter size: 1.15 MB


In [ ]:
# ============================================================================
# METHOD 3: QLORA (QUANTIZED LORA) - Simulated
# ============================================================================

print("\n" + "="*80)
print("METHOD 3: QLORA (QUANTIZED + LORA)")
print("="*80)

print("\n[6/7] Loading model for QLoRA fine-tuning...")

# Load fresh model
model_qlora = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=len(categories)
)
model_qlora.config.pad_token_id = tokenizer.pad_token_id

# Apply quantization (simulated)
def quantize_model(model):
    quantized_count = 0
    for name, param in model.named_parameters():
        if param.requires_grad and 'weight' in name and param.dim() >= 2:
            scales = param.abs().max(dim=1, keepdim=True)[0] / 127.0
            scales = scales.clamp(min=1e-5)
            param_quantized = (param / scales).round().clamp(-128, 127)
            param.data = param_quantized * scales
            quantized_count += param.numel()
    return quantized_count

quantized_count = quantize_model(model_qlora)
print(f"✓ Quantized {quantized_count:,} parameters to 8-bit")

# Apply LoRA
model_qlora = get_peft_model(model_qlora, lora_config)

print(f"✓ LoRA applied on top of quantized model")
model_qlora.print_trainable_parameters()

trainable_params_qlora = sum(p.numel() for p in model_qlora.parameters() if p.requires_grad)

# Training arguments for QLoRA
training_args_qlora = TrainingArguments(
    output_dir='./results_qlora',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    logging_steps=50,
)

trainer_qlora = Trainer(
    model=model_qlora,
    args=training_args_qlora,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting QLoRA fine-tuning...")
start_time_qlora = time.time()

train_result_qlora = trainer_qlora.train()

end_time_qlora = time.time()
qlora_train_time = end_time_qlora - start_time_qlora

# Evaluate
results_qlora = trainer_qlora.evaluate()

print("\n✓ QLoRA fine-tuning complete!")
print(f"  Training time: {qlora_train_time:.2f}s")
print(f"  Test accuracy: {results_qlora['eval_accuracy']:.4f}")
print(f"  Test F1: {results_qlora['eval_f1']:.4f}")

# Save model size
model_qlora.save_pretrained('./results_qlora/model')
qlora_model_size = sum(
    os.path.getsize(os.path.join('./results_qlora/model', f))
    for f in os.listdir('./results_qlora/model')
    if os.path.isfile(os.path.join('./results_qlora/model', f))
) / (1024**2)

print(f"  QLoRA adapter size: {qlora_model_size:.2f} MB")


METHOD 3: QLORA (QUANTIZED + LORA)

[6/7] Loading model for QLoRA fine-tuning...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Quantized 124,321,536 parameters to 8-bit
✓ LoRA applied on top of quantized model
trainable params: 297,984 || all params: 124,740,864 || trainable%: 0.2389

Starting QLoRA fine-tuning...


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.614400,0.584472,0.737620,0.741447,0.737620,0.733727
2,0.514500,0.524636,0.776792,0.775292,0.776792,0.775750
3,0.379600,0.525970,0.772358,0.770659,0.772358,0.770633


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# ============================================================================
# COMPREHENSIVE COMPARISON
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Method': ['Traditional Fine-Tuning', 'LoRA', 'QLoRA'],
    'Trainable Params': [
        f"{trainable_params_traditional:,} (100%)",
        f"{trainable_params_lora:,} ({100*trainable_params_lora/total_params:.2f}%)",
        f"{trainable_params_qlora:,} ({100*trainable_params_qlora/total_params:.2f}%)"
    ],
    'Training Time (s)': [
        f"{traditional_train_time:.2f}",
        f"{lora_train_time:.2f}",
        f"{qlora_train_time:.2f}"
    ],
    'Speedup': [
        "1.0x (baseline)",
        f"{traditional_train_time/lora_train_time:.2f}x",
        f"{traditional_train_time/qlora_train_time:.2f}x"
    ],
    'Model Size (MB)': [
        f"{traditional_model_size:.2f}",
        f"{lora_model_size:.2f}",
        f"{qlora_model_size:.2f}"
    ],
    'Size Reduction': [
        "0%",
        f"{100*(1-lora_model_size/traditional_model_size):.1f}%",
        f"{100*(1-qlora_model_size/traditional_model_size):.1f}%"
    ],
    'Test Accuracy': [
        f"{results_traditional['eval_accuracy']:.4f}",
        f"{results_lora['eval_accuracy']:.4f}",
        f"{results_qlora['eval_accuracy']:.4f}"
    ],
    'Test F1': [
        f"{results_traditional['eval_f1']:.4f}",
        f"{results_lora['eval_f1']:.4f}",
        f"{results_qlora['eval_f1']:.4f}"
    ],
})

print("\n")
print(comparison_df.to_string(index=False))